<a href="https://colab.research.google.com/github/kraszor/SIGK-2025Z/blob/main/transformacja-3d/sigk4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install trimesh

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import trimesh
import pandas as pd
from scipy.spatial.distance import cdist

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from google.colab import drive
drive.mount('/content/drive')

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
class Canonicalizer:
    """Class responsible for normalization of the object."""
    def __init__(self):
        self.centroid = None
        self.scale = None

    def fit_transform(self, mesh):
        vertices = mesh.vertices
        self.centroid = np.mean(vertices, axis=0)
        vertices = vertices - self.centroid
        self.scale = np.max(np.linalg.norm(vertices, axis=1))
        vertices = vertices / self.scale
        mesh.vertices = vertices
        return mesh

    def transform(self, mesh):
        vertices = mesh.vertices
        centroid = np.mean(vertices, axis=0)
        vertices = vertices - centroid
        scale = np.max(np.linalg.norm(vertices, axis=1))
        return vertices / scale


class PositionalEncoding(nn.Module):
    def __init__(self, num_freqs=6):
        super().__init__()
        self.num_freqs = num_freqs
        freq_bands = 2.0 ** torch.linspace(0.0, num_freqs - 1, num_freqs)
        self.register_buffer('freq_bands', freq_bands)

    def forward(self, x):
        embed_fns = []
        embed_fns.append(x)

        for freq in self.freq_bands:
            embed_fns.append(torch.sin(x * freq * np.pi))
            embed_fns.append(torch.cos(x * freq * np.pi))

        return torch.cat(embed_fns, dim=-1)

class DisplacementField(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_freqs = 3
        self.embedder = PositionalEncoding(num_freqs=self.num_freqs)

        input_dim = 3 + (3 * 2 * self.num_freqs)
        hidden_dim = 256
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.layer4 = nn.Linear(hidden_dim + input_dim, hidden_dim)
        self.layer5 = nn.Linear(hidden_dim, hidden_dim)
        self.layer6 = nn.Linear(hidden_dim, hidden_dim)
        self.out_layer = nn.Linear(hidden_dim, 3)

        self.act = nn.ReLU()
        self.out_act = nn.Tanh()

    def forward(self, x):
        x_emb = self.embedder(x)
        h = self.act(self.layer1(x_emb))
        h = self.act(self.layer2(h))
        h = self.act(self.layer3(h))

        h = torch.cat([h, x_emb], dim=-1)

        h = self.act(self.layer4(h))
        h = self.act(self.layer5(h))
        h = self.act(self.layer6(h))

        return self.out_act(self.out_layer(h))


In [4]:
def chamfer_distance(p1, p2):
    x = p1.unsqueeze(1)
    y = p2.unsqueeze(0)

    dist = torch.norm(x - y, dim=2)

    min_dist_p1, _ = torch.min(dist, dim=1)
    min_dist_p2, _ = torch.min(dist, dim=0)

    return torch.mean(min_dist_p1) + torch.mean(min_dist_p2)

In [5]:
from scipy.spatial import cKDTree

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def visualize_transformation(name, output_dir="./drive/MyDrive/sigk_4/results"):
    steps = [0, 50, 100]
    fig = plt.figure(figsize=(18, 6))
    fig.suptitle(f"Geometric transform (from files): {name} -> Teapot", fontsize=16)

    for i, step in enumerate(steps):
        filename = f"{output_dir}/{name}_step_{step}.obj"


        mesh = trimesh.load(filename)
        verts = mesh.vertices
        faces = mesh.faces

        ax = fig.add_subplot(1, 3, i + 1, projection='3d')

        ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2],
                        triangles=faces, cmap='viridis',
                        edgecolor='none', alpha=0.8)

        ax.set_title(f"Etap: {step}%")

        ax.set_xlim([-0.7, 0.7])
        ax.set_ylim([-0.7, 0.7])
        ax.set_zlim([-0.7, 0.7])
        ax.set_axis_off()
        ax.view_init(elev=20, azim=45)

    plt.tight_layout()
    plt.show()


def predict_in_batches(model, vertices, batch_size=10000):
    model.eval()
    n_verts = len(vertices)
    displacements = []

    with torch.no_grad():
        for i in range(0, n_verts, batch_size):
            batch_cpu = vertices[i : i + batch_size]
            batch_gpu = torch.tensor(batch_cpu, dtype=torch.float32).to(device)

            disp_gpu = model(batch_gpu)

            displacements.append(disp_gpu.cpu().numpy())

    return np.vstack(displacements)

def calculate_metrics_optimized(source_deformed, target_mesh, num_samples=5000):

    p_pred, _ = trimesh.sample.sample_surface(source_deformed, 2048)
    p_target, _ = trimesh.sample.sample_surface(target_mesh, 2048)

    tree_target = cKDTree(p_target)
    dist_pred_to_target, _ = tree_target.query(p_pred, k=1)
    tree_pred = cKDTree(p_pred)
    dist_target_to_pred, _ = tree_pred.query(p_target, k=1)
    chamfer = np.mean(dist_pred_to_target) + np.mean(dist_target_to_pred)


    try:
        lp_pred = source_deformed.simplify_quadratic_decimation(2000)
        lp_target = target_mesh.simplify_quadratic_decimation(2000)
    except Exception:
        lp_pred = source_deformed
        lp_target = target_mesh


    bbox_min = np.min([lp_pred.bounds[0], lp_target.bounds[0]], axis=0)
    bbox_max = np.max([lp_pred.bounds[1], lp_target.bounds[1]], axis=0)
    test_points = np.random.uniform(bbox_min, bbox_max, (num_samples, 3))

    contains_pred = lp_pred.contains(test_points)
    contains_target = lp_target.contains(test_points)

    intersection = np.sum(contains_pred & contains_target)
    union = np.sum(contains_pred | contains_target)

    jaccard = intersection / union if union > 0 else 0.0
    dice = 2 * intersection / (np.sum(contains_pred) + np.sum(contains_target)) if union > 0 else 0.0

    return chamfer, jaccard, dice

def save_and_evaluate(model, source_mesh, target_mesh, name, output_dir="./drive/MyDrive/sigk_4/results", generate_models=True):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Generate results for: {name}...")

    dense_source = source_mesh.copy()

    dense_source = dense_source.subdivide()

    verts_cpu = dense_source.vertices
    final_displacement = predict_in_batches(model, verts_cpu, batch_size=5000)

    steps = [0.0, 0.5, 1.0]
    final_deformed_mesh = None

    for t in steps:
      if(generate_models or t == 1.0):
        new_verts = verts_cpu + (final_displacement * t)
        mesh_copy = dense_source.copy()
        mesh_copy.vertices = new_verts

        filename = f"{output_dir}/{name}_step_{int(t*100)}.obj"
        mesh_copy.export(filename)
        print(f"Saved file {filename}")

        if t == 1.0:
            final_deformed_mesh = mesh_copy

    chamfer, jaccard, dice = calculate_metrics_optimized(final_deformed_mesh, target_mesh)

    return {
        "Metoda": name,
        "IoU": jaccard,
        "Dice": dice,
        "Chamfer": chamfer
    }

In [6]:
def train_deformation(source_name, source_path, target_path, epochs=1000, lr=0.001):
    print(f"Starting training: {source_name} -> Teapot")

    s_mesh = trimesh.load(source_path)
    t_mesh = trimesh.load(target_path)

    canon = Canonicalizer()
    s_mesh = canon.fit_transform(s_mesh)
    t_mesh = Canonicalizer().fit_transform(t_mesh)

    t_points_np, _ = trimesh.sample.sample_surface(t_mesh, 10000)
    t_points = torch.tensor(t_points_np, dtype=torch.float32).to(device)

    model = DisplacementField().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                     factor=0.5, patience=50)

    model.train()

    for i in range(epochs):
        optimizer.zero_grad()

        s_points_np, _ = trimesh.sample.sample_surface(s_mesh, 10000)
        s_points = torch.tensor(s_points_np, dtype=torch.float32).to(device)

        displacement = model(s_points)
        deformed_points = s_points + displacement

        chamfer = chamfer_distance(deformed_points, t_points)

        smoothness_loss = torch.mean(displacement ** 2)

        loss = chamfer + (0.15 * smoothness_loss)

        loss.backward()
        optimizer.step()

        scheduler.step(loss)

        if i % 100 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {i}/{epochs} | Loss: {loss.item():.6f} | LR: {current_lr:.6f}")

    print("Training finished")
    return model, s_mesh, t_mesh

Data paths

In [7]:
DATA_DIR = "./drive/MyDrive/SIGK/3D_transformation"
SAVE_DIR = "./drive/MyDrive/SIGK/3D_transformation/results"

train_model = False

TARGET_FILE = os.path.join(DATA_DIR, "teapot.obj")

OBJECTS = {
"Bunny": os.path.join(DATA_DIR, "bunny.obj"),
"Dragon": os.path.join(DATA_DIR, "dragon.obj"),
"Armadillo": os.path.join(DATA_DIR, "armadillo.obj")
}

ASIAN_DRAGON_FILE = os.path.join(DATA_DIR, "asian_dragon_small.obj")

Train models

In [8]:
import gc
if train_model:
  for name, path in OBJECTS.items():
    model, s_mesh, t_mesh = train_deformation(name, path, TARGET_FILE, epochs=1100)

    if model:
        model_path = os.path.join(SAVE_DIR, f"model_{name.lower()}.pth")
        torch.save(model.state_dict(), model_path)
        print(f"Saved model weights for {name}: {model_path}")
        visualize_transformation(name, SAVE_DIR)

    del s_mesh
    del t_mesh
    torch.cuda.empty_cache()
    gc.collect()


Evaluate models

In [ ]:
results_list = []
for name, source_path in OBJECTS.items():
    model_path = os.path.join(SAVE_DIR, f"model_{name.lower()}.pth")

    if os.path.exists(model_path):
      model = DisplacementField().to(device)
      model.load_state_dict(torch.load(model_path, map_location=device))
      model.eval()

      s_mesh = trimesh.load(source_path)
      t_mesh = trimesh.load(TARGET_FILE)

      model = DisplacementField().to(device)

      state_dict = torch.load(model_path, map_location=device)

      model.load_state_dict(state_dict)

      model.eval()

      print(f"Loaded model: {name} from {model_path}")

      metrics = save_and_evaluate(model, s_mesh, t_mesh, name, SAVE_DIR)
      results_list.append(metrics)
      print(metrics)

      #visualize_transformation(name, SAVE_DIR)

      del s_mesh
      torch.cuda.empty_cache()
      gc.collect()


df_results = pd.DataFrame(results_list)
print("\n --- Summary ---")
print(df_results.to_string(index=False))

Loaded model: Bunny from ./drive/MyDrive/SIGK/3D_transformation/results/model_bunny.pth
Generate results for: Bunny...
Saved file ./drive/MyDrive/SIGK/3D_transformation/results/Bunny_step_0.obj
Saved file ./drive/MyDrive/SIGK/3D_transformation/results/Bunny_step_50.obj
Saved file ./drive/MyDrive/SIGK/3D_transformation/results/Bunny_step_100.obj
{'Metoda': 'Bunny', 'IoU': np.float64(0.0), 'Dice': np.float64(0.0), 'Chamfer': np.float64(2.3450489852007643)}
Loaded model: Dragon from ./drive/MyDrive/SIGK/3D_transformation/results/model_dragon.pth
Generate results for: Dragon...
Saved file ./drive/MyDrive/SIGK/3D_transformation/results/Dragon_step_0.obj
Saved file ./drive/MyDrive/SIGK/3D_transformation/results/Dragon_step_50.obj
Saved file ./drive/MyDrive/SIGK/3D_transformation/results/Dragon_step_100.obj
{'Metoda': 'Dragon', 'IoU': np.float64(0.0), 'Dice': np.float64(0.0), 'Chamfer': np.float64(2.299865086522308)}
Loaded model: Armadillo from ./drive/MyDrive/SIGK/3D_transformation/results/

Asian Dragon experiment

In [ ]:
if os.path.exists(ASIAN_DRAGON_FILE):
    asian_mesh = trimesh.load(ASIAN_DRAGON_FILE)

    canon = Canonicalizer()
    asian_mesh = canon.fit_transform(asian_mesh)

    t_mesh = trimesh.load(TARGET_FILE)
    t_mesh = Canonicalizer().fit_transform(t_mesh)

    for name, source_path in OBJECTS.items():
        model_path = os.path.join(SAVE_DIR, f"model_{name.lower()}.pth")

        if os.path.exists(model_path):
          test_name = f"{name}-flow_asian_dragon"

          print(f"Evaluate asian dragon: {test_name}")
          model = DisplacementField().to(device)
          model.load_state_dict(torch.load(model_path, map_location=device))
          model.eval()

          metrics = save_and_evaluate(model, asian_mesh, t_mesh, test_name, SAVE_DIR)
          results_list.append(metrics)

          #visualize_transformation(test_name, SAVE_DIR)

          torch.cuda.empty_cache()
          gc.collect()

df_results = pd.DataFrame(results_list)
print("\n --- Summary ---")
print(df_results.to_string(index=False))